![header](content/img/header.PNG)

# Part 4 — Anticipating wet conditions in Morocco using GloFAS forecasts

In Part 3, we used GloFAS historical reanalysis to describe historical river discharge conditions. In this notebook, we move from historical analysis to forecast interpretation.

The scientific question is:

> **Could GloFAS provide early indications that a prolonged wet and flood-prone period was developing in Morocco?**

This notebook uses the GloFAS medium-range **control forecast** for January–April 2026. The objective is not formal forecast verification. Instead, we show how daily forecasts can be interpreted against daily historical reference conditions for the same time of year.

```{admonition} Learning objectives
:class: tip

By the end of this notebook, you will be able to:

- retrieve GloFAS medium-range control forecasts from EWDS;
- open one NetCDF forecast file per forecast issue date;
- compare daily forecast values with daily historical reference conditions;
- identify forecast values above the historical daily Q90 threshold;
- inspect one 30-day forecast hydrograph at a representative basin;
- map where the forecast repeatedly indicated unusually high discharge;
- examine whether the wet signal persisted across successive forecast cycles.
```


## 1. Download GloFAS medium-range control forecasts

This section is included for reproducibility. If the forecast files already exist locally, the loop skips them. If they are missing, one NetCDF file is retrieved for each forecast issue date.

Each file contains a 30-day control forecast.


In [ ]:
# === Retrieve GloFAS medium-range control forecasts for Morocco ===

import datetime
from pathlib import Path
import cdsapi

DATASET = "cems-glofas-forecast"

# Forecast issue dates to retrieve
START_DATE = "2026-01-01"
END_DATE = "2026-04-30"

# 30 daily lead times: +1 day to +30 days
LEADTIMES = [str(lt) for lt in range(24, 744, 24)]

# Morocco domain [North, West, South, East]
BBOX_MOROCCO = [36.20, -17.50, 20.50, -0.50]

DATA_DIR = Path("/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/")
FORECAST_DIR = DATA_DIR / "forecast/control"
FORECAST_DIR.mkdir(parents=True, exist_ok=True)


def compute_dates_range(start_date, end_date):
    start = datetime.date.fromisoformat(start_date)
    end = datetime.date.fromisoformat(end_date)
    return [start + datetime.timedelta(days=i) for i in range((end - start).days + 1)]


client = cdsapi.Client()

for date in compute_dates_range(START_DATE, END_DATE):
    year = date.strftime("%Y")
    month = date.strftime("%m")
    day = date.strftime("%d")

    target = FORECAST_DIR / f"{DATASET}_{year}_{month}_{day}_morocco_control.nc"

    if target.exists():
        print(f"Already exists, skipping: {target.name}")
        continue

    request = {
        "system_version": "operational",
        "hydrological_model": "lisflood",
        "product_type": "ensemble_perturbed_forecasts",
        "variable": "river_discharge_in_the_last_24_hours",
        "year": year,
        "month": month,
        "day": day,
        "leadtime_hour": LEADTIMES,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": BBOX_MOROCCO,
    }

    print(f"Retrieving: {target.name}")
    client.retrieve(DATASET, request, str(target))


## 2. Configure the analysis

The analysis focuses on one representative basin in northern Morocco and a small surrounding domain. The local domain keeps the calculations responsive while preserving the spatial context around the basin.

The forecast comparison uses daily historical discharge from **1979–2025** as the reference period. The 2026 values are kept aside to describe what happened during the application period.


In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import warnings

DATA_DIR = Path("/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/")

# Daily historical discharge used to build daily reference conditions.
HIST_DAILY_FILE = DATA_DIR / "cems-glofas-historical_morocco_1979_2026.nc"

# Daily control forecast files: one NetCDF file per forecast issue date.
FORECAST_DIR = DATA_DIR / "forecast/control"
FORECAST_PATTERN = "cems-glofas-forecast_*.nc"

# Study basin.
LAT_BASIN = 35.2327
LON_BASIN = -6.1209 + 360

# Local analysis domain around the basin.
# Latitude is stored north-to-south in the files.
LAT_MIN = 34.50
LAT_MAX = 36.00
LON_MIN = 352.80
LON_MAX = 354.80

# Historical reference period.
REF_START = "1979-01-01"
REF_END = "2025-12-31"

# Period used to describe what happened in 2026.
APP_START = "2026-01-01"
APP_END = "2026-04-30"

warnings.filterwarnings("ignore", message="All-NaN slice encountered")

print("Daily historical file:", HIST_DAILY_FILE)
print("Forecast directory   :", FORECAST_DIR)
print(f"Study basin          : lat={LAT_BASIN:.4f}, lon={LON_BASIN:.4f}")
print(f"Local domain         : lat {LAT_MIN}–{LAT_MAX}, lon {LON_MIN}–{LON_MAX}")


Daily historical file: /perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/cems-glofas-historical_morocco_1979_2026.nc
Forecast directory   : /perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/forecast/control
Study basin          : lat=35.2327, lon=353.8791
Local domain         : lat 34.5–36.0, lon 352.8–354.8


## 3. Build daily historical reference conditions

Forecast discharge is a daily value, so the reference conditions must also be daily.

For each day of the year, we calculate:

- the historical daily mean;
- the daily Q10 threshold;
- the daily Q90 threshold.

For example, a forecast valid on 15 February is compared with historical 15 February values from 1979–2025.


In [2]:
# Open daily historical discharge.
ds_daily = xr.open_dataset(HIST_DAILY_FILE)
dis_daily = ds_daily["dis24"]

# Select the local domain before calculating reference statistics.
dis_daily_area = dis_daily.sel(
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX),
)

# Historical discharge during the application period.
hist_2026_daily_area = dis_daily_area.sel(
    valid_time=slice(APP_START, APP_END)
)

# Reference period used to build daily climatological statistics.
reference_daily_area = dis_daily_area.sel(
    valid_time=slice(REF_START, REF_END)
)

# Load the local domain into memory once.
# This is much smaller than the full Morocco grid.
reference_daily_area = reference_daily_area.load()

# Daily reference statistics by day of year.
daily_mean_area = reference_daily_area.groupby("valid_time.dayofyear").mean(
    dim="valid_time",
    skipna=True,
)

daily_q10_area = reference_daily_area.groupby("valid_time.dayofyear").quantile(
    0.10,
    dim="valid_time",
    skipna=True,
)

daily_q90_area = reference_daily_area.groupby("valid_time.dayofyear").quantile(
    0.90,
    dim="valid_time",
    skipna=True,
)

print("Daily historical subset:")
print(dis_daily_area)
print("Daily reference mean:")
print(daily_mean_area)
print("Daily Q10 reference:")
print(daily_q10_area)
print("Daily Q90 reference:")
print(daily_q90_area)


Daily historical subset:
<xarray.DataArray 'dis24' (valid_time: 17335, latitude: 30, longitude: 40)> Size: 83MB
[20802000 values with dtype=float32]
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 139kB 1979-01-02 ... 2026-06-18
  * latitude    (latitude) float64 240B 35.98 35.93 35.88 ... 34.63 34.58 34.53
  * longitude   (longitude) float64 320B 352.8 352.9 352.9 ... 354.7 354.7 354.8
    surface     float64 8B 0.0
Attributes: (12/32)
    GRIB_paramId:                             240024
    GRIB_dataType:                            sfo
    GRIB_numberOfPoints:                      106760
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            avg
    ...                                       ...
    GRIB_name:                                Mean discharge in the last 24 h...
    GRIB_shortName:                           dis24
    GRIB_units:                               m**3 s**-1
  

## 4. Open the control forecasts

The forecast archive contains one NetCDF file per forecast issue date.

Each forecast has three time-related coordinates:

- `forecast_reference_time`: when the forecast was issued;
- `forecast_period`: the lead time, from +1 to +30 days;
- `valid_time`: the calendar date represented by each forecast value.

The files are already downloaded and combined into one file under : Data/glofas_control_forecast_morocco_jan_apr2026.nc

In [5]:
# Import and inspect the file
FORECAST_FILE = DATA_DIR / "glofas_control_forecast_morocco_jan_apr2026.nc"

ds_fc = xr.open_dataset(FORECAST_FILE)

fc_area = ds_fc["dis24"]

print(ds_fc)

<xarray.Dataset> Size: 17MB
Dimensions:                  (forecast_period: 30,
                              forecast_reference_time: 120, latitude: 30,
                              longitude: 40)
Coordinates:
  * forecast_period          (forecast_period) timedelta64[ns] 240B 1 days .....
    valid_time               (forecast_reference_time, forecast_period) datetime64[ns] 29kB ...
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 960B 20...
  * latitude                 (latitude) float64 240B 35.98 35.93 ... 34.58 34.53
  * longitude                (longitude) float64 320B 352.8 352.9 ... 354.8
    number                   int64 8B ...
    surface                  float64 8B ...
Data variables:
    dis24                    (forecast_period, forecast_reference_time, latitude, longitude) float32 17MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    G

## 5. Compare forecasts with daily historical conditions

Each forecast value is matched to the historical reference for the same day of the year.

This produces two forecast diagnostics:

1. **daily anomaly**: forecast discharge minus the historical daily mean;
2. **daily Q90 exceedance**: forecast discharge above the historical daily Q90 threshold.

These diagnostics show whether forecast river discharge is unusual for that time of year.


In [ ]:
# Match daily historical references to the forecast grid.
daily_mean_on_fc_grid = daily_mean_area.sel(
    latitude=ds_fc.latitude,
    longitude=ds_fc.longitude,
    method="nearest",
)

daily_q10_on_fc_grid = daily_q10_area.sel(
    latitude=ds_fc.latitude,
    longitude=ds_fc.longitude,
    method="nearest",
)

daily_q90_on_fc_grid = daily_q90_area.sel(
    latitude=ds_fc.latitude,
    longitude=ds_fc.longitude,
    method="nearest",
)

# Select the historical reference for the forecast valid date.
valid_doy = ds_fc.valid_time.dt.dayofyear
mean_for_fc = daily_mean_on_fc_grid.sel(dayofyear=valid_doy)
q90_for_fc = daily_q90_on_fc_grid.sel(dayofyear=valid_doy)

forecast_anomaly = fc_area - mean_for_fc
forecast_above_q90 = fc_area > q90_for_fc

print("Daily forecast anomaly:")
print(forecast_anomaly)
print("Forecast above daily Q90:")
print(forecast_above_q90)


## 6. Analyse one forecast cycle at the basin

We first inspect one forecast run at the representative basin. This is the most direct way to understand what the forecast was indicating.

The 30-day forecast is compared with the historical daily mean and the Q10–Q90 reference range for the same valid dates.


In [ ]:
# Select one forecast issue date.
ISSUE_DATE = np.datetime64("2026-02-01")

ds_one = ds_fc.sel(
    forecast_reference_time=ISSUE_DATE,
    method="nearest",
)

fc_one = ds_one["dis24"]

# Extract forecast at the basin point.
fc_one_basin = fc_one.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
)

valid_time_basin = ds_one["valid_time"]
forecast_doy = valid_time_basin.dt.dayofyear

# Extract daily references at the basin point for the same valid dates.
mean_basin = daily_mean_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=forecast_doy)

q10_basin = daily_q10_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=forecast_doy)

q90_basin = daily_q90_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=forecast_doy)

# Load values for plotting.
fc_one_basin = fc_one_basin.compute()
mean_basin = mean_basin.compute()
q10_basin = q10_basin.compute()
q90_basin = q90_basin.compute()
valid_time_basin = valid_time_basin.compute()

print("Selected forecast issue date:", pd.to_datetime(ds_one.forecast_reference_time.values).date())
print("Selected forecast grid point:", float(fc_one_basin.latitude.values), float(fc_one_basin.longitude.values))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.fill_between(
    valid_time_basin.values,
    q10_basin.values,
    q90_basin.values,
    alpha=0.18,
    label="Historical Q10–Q90 range",
)

ax.plot(
    valid_time_basin.values,
    mean_basin.values,
    label="Historical daily mean",
    linestyle="--",
    linewidth=1.5,
)

ax.plot(
    valid_time_basin.values,
    q90_basin.values,
    label="Historical daily Q90",
    linestyle=":",
    linewidth=1.5,
)

ax.plot(
    valid_time_basin.values,
    fc_one_basin.values,
    label="GloFAS control forecast",
    linewidth=2.2,
)

ax.set_title("One 30-day GloFAS forecast at the study basin")
ax.set_xlabel("Forecast valid date")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

plt.tight_layout()
plt.show()


When the forecast line rises above Q90, it indicates discharge values that are high compared with the historical distribution for those calendar days.


## 7. Regional overview of the wet signal

After inspecting one forecast cycle, we summarise the signal over the local domain.

For each grid cell, we calculate the percentage of forecast values that exceed the historical daily Q90 threshold. This is not an ensemble probability: it is a measure of how often the deterministic control forecast indicated unusually high discharge across the available forecast issue dates and lead times.


In [ ]:
q90_signal_frequency = (
    forecast_above_q90
    .mean(dim=("forecast_reference_time", "forecast_period"), skipna=True)
    * 100
).compute()

print("Daily Q90 signal frequency (%):")
display(q90_signal_frequency)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

q90_signal_frequency.plot(
    ax=ax,
    x="longitude",
    y="latitude",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    cbar_kwargs={"label": "Forecast values above daily Q90 (%)"},
)

ax.scatter(
    LON_BASIN,
    LAT_BASIN,
    marker="x",
    s=80,
    color="black",
    label="Study basin",
)

ax.set_title("Persistence of daily Q90 exceedance signal")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()

plt.tight_layout()
plt.show()


High values indicate places where the control forecast repeatedly exceeded the historical daily Q90 threshold. Spatially coherent areas of high frequency suggest a persistent wet signal in the forecast.


## 8. Forecast evolution at the basin

Forecasters do not usually rely on a single run. They look for consistency across successive forecast cycles.

Here, we calculate the percentage of lead days above Q90 for each forecast issue date at the study basin. This helps identify whether the wet signal persisted over time.


In [ ]:
# Extract the forecast and Q90 threshold at the basin point.
fc_basin = fc_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
)

q90_basin_for_fc = q90_for_fc.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
)

above_q90_basin = fc_basin > q90_basin_for_fc

basin_signal_by_issue_date = (
    above_q90_basin
    .mean(dim="forecast_period", skipna=True)
    * 100
).compute()

print(basin_signal_by_issue_date)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(
    basin_signal_by_issue_date.forecast_reference_time,
    basin_signal_by_issue_date,
    marker="o",
    linewidth=1.5,
)

ax.set_title("Persistence of the wet signal at the study basin")
ax.set_ylabel("Lead days above daily Q90 (%)")
ax.set_xlabel("Forecast issue date")
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()


## 9. Inspect selected forecast cycles

We now plot a few selected 30-day forecast trajectories at the basin. These are compared with the historical daily reference range and the historical 2026 discharge.

This view helps answer whether successive forecasts were telling a consistent hydrological story.


In [ ]:
# Daily references at the basin point.
daily_mean_basin = daily_mean_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
)

daily_q10_basin = daily_q10_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
)

daily_q90_basin = daily_q90_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
)

hist_2026_basin = hist_2026_daily_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).compute()

# Build a table for historical 2026 values and matching daily references.
hist_2026_table = hist_2026_basin.to_dataframe(name="historical_2026").reset_index()
hist_2026_table["dayofyear"] = hist_2026_table["valid_time"].dt.dayofyear

hist_2026_table["daily_mean"] = hist_2026_table["dayofyear"].map(
    dict(zip(daily_mean_basin.dayofyear.values, daily_mean_basin.values))
)

hist_2026_table["q10"] = hist_2026_table["dayofyear"].map(
    dict(zip(daily_q10_basin.dayofyear.values, daily_q10_basin.values))
)

hist_2026_table["q90"] = hist_2026_table["dayofyear"].map(
    dict(zip(daily_q90_basin.dayofyear.values, daily_q90_basin.values))
)

hist_2026_table.head()


In [ ]:
SELECTED_ISSUE_DATES = [
    "2026-01-05",
    "2026-01-15",
    "2026-02-01",
    "2026-02-15",
    "2026-03-01",
]

fig, ax = plt.subplots(figsize=(13, 5))

for issue_date in SELECTED_ISSUE_DATES:
    ref_time = np.datetime64(issue_date)

    if ref_time not in ds_fc.forecast_reference_time.values:
        print(f"Issue date not available, skipping: {issue_date}")
        continue

    y = fc_basin.sel(forecast_reference_time=ref_time).compute()
    x = ds_fc.valid_time.sel(forecast_reference_time=ref_time).compute()

    ax.plot(x, y, linewidth=1.5, label=f"Forecast issued {issue_date}")

# Historical 2026 values.
ax.plot(
    hist_2026_table["valid_time"],
    hist_2026_table["historical_2026"],
    color="black",
    linewidth=2.0,
    label="Historical daily 2026",
    zorder=5,
)

# Daily reference conditions for Jan-April.
ax.plot(
    hist_2026_table["valid_time"],
    hist_2026_table["daily_mean"],
    color="grey",
    linewidth=1.5,
    linestyle="--",
    label="Historical daily mean",
)

ax.fill_between(
    hist_2026_table["valid_time"],
    hist_2026_table["q10"],
    hist_2026_table["q90"],
    color="lightgrey",
    alpha=0.5,
    label="Daily Q10–Q90 range",
)

ax.set_title("Selected GloFAS control forecasts at the study basin")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.set_xlabel("Valid date")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()


## 10. Interpretation

This notebook demonstrates how GloFAS can support early awareness of developing wet and flood-prone conditions.

The main comparison is made at the daily scale: each forecast value is compared with historical conditions for the same day of the year. This keeps the forecast and the reference conditions on the same temporal scale.

The key outputs are:

- a 30-day forecast hydrograph at the study basin;
- a map of repeated daily Q90 exceedance within the local domain;
- a time series showing whether the basin-scale wet signal persisted across forecast cycles;
- selected forecast trajectories compared with the historical daily reference range.

Together, these products answer the operational question:

> **Was GloFAS repeatedly indicating above-normal river flows during the January–April wet period?**


## 11. Extension: perturbed forecasts

When perturbed ensemble forecasts are available for the same period, the workflow can be extended without changing the scientific structure.

The deterministic signal shown here can become an ensemble-based confidence assessment using:

- spaghetti plots for the study basin;
- boxplots by valid date;
- probability of exceeding daily Q90;
- probability of exceeding flood thresholds, if return-period thresholds are added.

For example, instead of asking:

> Did the control forecast exceed Q90?

we can ask:

> What percentage of ensemble members exceeded Q90?


## Exercises

```{exercise}
Change the study basin coordinates and repeat the basin-scale analysis.
```

```{exercise}
Identify the forecast issue dates with the strongest Q90 signal at the study basin.
```

```{exercise}
Compare two basins within the local domain. Which basin shows the more persistent wet signal?
```
